In [1]:
import pennylane as qml
from matplotlib import pyplot as plt
from pennylane import numpy as np
import scipy
import networkx as nx
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
import copy
import os
import random
import torch
import itertools

In [ ]:
adata=sc.read_h5ad("scRNAseq_with_latent_embedding.h5ad")

In [ ]:
true_label = adata.obs['annotation_level_3'].values 
# Now run VIA with the specific root cell
g = via.VIA(
    adata.obsm["X_pca"],
    true_label=true_label,
    knn=50,
    root_user=[13805],  # Use specific cell index, in our case, OPC-like is the root cell
    dataset='group',
    random_seed=42,
    jac_weighted_edges=True,
    keep_all_local_dist=False,
    user_defined_terminal_group=['AC-like', 'MES-like'],
    preserve_disconnected=False,
)
g.run_VIA(),

adata.obs["VIA_pseudotime2"] = g.single_cell_pt_markov
adata.obs["VIA_cluster2"] = g.labels

genes_of_interest = [
    "ENSG00000132692",
    "ENSG00000117632",
    "ENSG00000144485",
    "ENSG00000006468",
    "ENSG00000175161",
    "ENSG00000156103",
    "ENSG00000166165",
    "ENSG00000050405",
    "ENSG00000038427",
    "ENSG00000189159",
    "ENSG00000139352",
    "ENSG00000135446",
    "ENSG00000137285",
    "ENSG00000149294"
]


# Subset to genes of interest
expr = pd.DataFrame(
    adata[:, genes_of_interest].X.toarray() if hasattr(adata[:, genes_of_interest].X, "toarray") else adata[:, genes_of_interest].X,
    index=adata.obs_names,
    columns=genes_of_interest
).T  # genes x cells

# Append pseudotime as final row
pseudotime_row = pd.DataFrame(
    adata.obs["VIA_pseudotime2"].values.reshape(1, -1),
    index=["VIA_pseudotime2"],
    columns=adata.obs_names
)

result = pd.concat([expr, pseudotime_row])
result.to_csv("/path/to/output/lognorm_expression_by_pseudotime.csv")


In [ ]:

csv_path = "path/to/output/lognorm_expression_by_pseudotime.csv"
df = pd.read_csv(csv_path, index_col=0)


pseudotime_row = df.iloc[[14]]     # double brackets to keep it as DataFrame
df_rest = df.drop(index=df.index[14])

# Apply MinMax scaling to the rest
scaler = MinMaxScaler()
df_scaled_rest = pd.DataFrame(scaler.fit_transform(df_rest), 
                              columns=df.columns,
                              index=df_rest.index)
arr = df_scaled_rest.to_numpy()

# Define conditions and choices
conditions = [
    (arr < 0.1585),
    (arr >= 0.1585) & (arr < 0.5),
    (arr >= 0.5) & (arr < 0.8415),
    (arr >= 0.8415)
]
choices = [0, 1, 2, 3]

 

# Apply the transformation
df_scaled_rest = pd.DataFrame(np.select(conditions, choices, default=0).astype(np.int32), index=df_scaled_rest.index, columns=df_scaled_rest.columns)
df = pd.concat([df_scaled_rest, pseudotime_row])

 
df


In [ ]:


# Assume df is your input dataframe (14 genes + 1 pseudotime row)
genes_df = df.iloc[:14, :]
pseudotime = df.iloc[14, :]

# Step 1: Assign each cell to 25 bins based on pseudotime
bin_labels = range(25)
bins = pd.qcut(pseudotime, q=25, labels=bin_labels)

# Step 2: Prepare output
compressed_data = []
pseudotime_medians = []

# Step 3: Iterate over bins
for bin_id in bin_labels:
    # Get indices (column names) of cells in this bin
    bin_cols = pseudotime.index[bins == bin_id]
    if len(bin_cols) == 0:
        continue  # skip empty bins

    # Subset first 6 genes for these columns (row by position, col by label)
    subset = df.iloc[:6, :].loc[:, bin_cols]



    # Convert each column into base-4 compressed integer
    base4_weights = np.array([4**i for i in reversed(range(6))])  # [1024, 256, 64, 16, 4, 1]
    values = subset.to_numpy().astype(int)
    compressed = values.T @ base4_weights  # shape: (num_cells_in_bin, )

    # Get median pseudotime of this bin
    median_pt = pseudotime[bin_cols].median()

    compressed_data.append(compressed)
    pseudotime_medians.append(median_pt)

# Step 4: Make output dataframe
output_df = pd.DataFrame(compressed_data, index=pseudotime_medians)

# Optional: sort by pseudotime
output_df = output_df.sort_index()
output_df 